In [1]:
import numpy as np

In [14]:

t = np.ones((10, 1))
b = np.pad(t, pad_width=((1,1), (0,0)), mode='constant', constant_values=0)
b[-1]

array([0.])

In [ ]:
from numba import njit
import timeit

def _TDMA(a, b, c, d, x):
    for i in range(1, len(b)):
        m = a[i] / b[i-1]
        b[i] = b[i] - m * c[i-1]
        d[i] = d[i] - m * d[i-1]
    x[-1] = d[-1] / b[-1]
    for i in range(len(x) -2, -1, -1):
        x[i] = (d[i] - c[i] * x[i+1]) / b[i]
    
    return x

_TDMA_jit = njit(_TDMA)

In [39]:
# Lower diagonal (a)
from numpy import zeros_like


lower = np.array([0., -1., -1., -1., -1.], dtype='float64')

# Main diagonal (b)
main = np.array([4., 4., 4., 4., 4.], dtype='float64')

# Upper diagonal (c)
upper = np.array([-1., -1., -1., -1., 0.], dtype='float64')

x = zeros_like(upper)
# Right-hand side (d)
rhs = np.array([2., 4., 6., 8., 16.], dtype='float64')

In [44]:
A = (
    np.diag(main)
    + np.diag(upper[:-1], 1)
    + np.diag(lower[1:], -1)
)

x_tdma = _TDMA(
    lower.copy(),
    main.copy(),
    upper.copy(),
    rhs.copy(),
    np.zeros_like(rhs)
)

x_numpy = np.linalg.solve(A, rhs)

print("TDMA solution:", x_tdma)
print("NumPy solution:", x_numpy)
print("Residual:", np.linalg.norm(A @ x_tdma - rhs))

18.660287081339714
3.7320574162679425
TDMA solution: [1. 2. 3. 4. 5.]
NumPy solution: [1. 2. 3. 4. 5.]
Residual: 1.7763568394002505e-15
